# Task 1 - Data Extraction

This notebook extracts data from:
- OpenSky API
- Open-Meteo API
- Aircraft reference dataset

The extracted raw data is saved in `data/raw/`.
Extraction activity is recorded in `logs/extraction_log.csv`.


In [1]:
print("kernel works")

kernel works


In [2]:
import requests
import json
import pandas as pd

from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter


In [3]:
# Resolve project paths safely
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

raw_path = PROJECT_ROOT / "data" / "raw"
logs_path = PROJECT_ROOT / "logs"

raw_path.mkdir(parents=True, exist_ok=True)
logs_path.mkdir(parents=True, exist_ok=True)

# One timestamp for this extraction run
extraction_time = datetime.now(timezone.utc)
timestamp = extraction_time.strftime("%Y-%m-%d_%H-%M-%S")

print("Raw data folder is ready.")
print("Logs folder is ready.")
print("Extraction time (UTC):", extraction_time)


Raw data folder is ready.
Logs folder is ready.
Extraction time (UTC): 2026-09-18 15:00:19.613684+00:00


In [4]:
# Extraction log file
log_file = logs_path / "extraction_log.csv"


def write_extraction_log(
    source,
    status,
    rows,
    file_size_mb,
    duration_seconds,
    status_code=None,
    error_message=None
):
    log_row = {
        "extraction_time_utc": extraction_time.isoformat(),
        "source": source,
        "status": status,
        "rows_extracted": rows,
        "file_size_mb": file_size_mb,
        "duration_seconds": round(duration_seconds, 2),
        "status_code": status_code,
        "error_message": error_message
    }

    log_df = pd.DataFrame([log_row])

    if log_file.exists():
        log_df.to_csv(log_file, mode="a", header=False, index=False)
    else:
        log_df.to_csv(log_file, index=False)


## 1. Extract OpenSky Data

Source: OpenSky Network

The request uses a geographic bounding box around Saudi Arabia to reduce the response size. Exact Saudi boundary filtering will be performed later using a polygon.

`extended=1` is retained so the extended state vector field is returned.


In [5]:
opensky_url = "https://opensky-network.org/api/states/all"

# Bounding box around Saudi Arabia
params = {
    "lamin": 16.0,
    "lomin": 34.0,
    "lamax": 33.0,
    "lomax": 56.0,
    "extended": 1
}

start_time = perf_counter()
states = []
opensky_data = None

try:
    response = requests.get(
        opensky_url,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    opensky_data = response.json()
    duration = perf_counter() - start_time

    states = opensky_data.get("states") or []
    aircraft_count = len(states)

    opensky_file = raw_path / f"opensky_{timestamp}.json"

    with open(opensky_file, "w", encoding="utf-8") as f:
        json.dump(opensky_data, f)

    file_size_mb = opensky_file.stat().st_size / (1024 * 1024)

    print("OpenSky Status Code:", response.status_code)
    print("Number of aircraft:", aircraft_count)
    print("File size:", round(file_size_mb, 3), "MB")
    print("Request duration:", round(duration, 2), "seconds")
    print("OpenSky data saved to:", opensky_file)

    write_extraction_log(
        source="OpenSky",
        status="SUCCESS",
        rows=aircraft_count,
        file_size_mb=round(file_size_mb, 3),
        duration_seconds=duration,
        status_code=response.status_code
    )

except requests.exceptions.Timeout:
    duration = perf_counter() - start_time
    print("OpenSky request timed out.")

    write_extraction_log(
        source="OpenSky",
        status="FAILED",
        rows=0,
        file_size_mb=0,
        duration_seconds=duration,
        error_message="Request timed out"
    )

except requests.exceptions.HTTPError as e:
    duration = perf_counter() - start_time
    status_code = response.status_code if "response" in locals() else None
    print("OpenSky HTTP error:", e)

    write_extraction_log(
        source="OpenSky",
        status="FAILED",
        rows=0,
        file_size_mb=0,
        duration_seconds=duration,
        status_code=status_code,
        error_message=str(e)
    )

except requests.exceptions.RequestException as e:
    duration = perf_counter() - start_time
    print("OpenSky request failed:", e)

    write_extraction_log(
        source="OpenSky",
        status="FAILED",
        rows=0,
        file_size_mb=0,
        duration_seconds=duration,
        error_message=str(e)
    )


OpenSky Status Code: 200
Number of aircraft: 131
File size: 0.019 MB
Request duration: 0.88 seconds
OpenSky data saved to: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\raw\opensky_2026-09-18_15-00-19.json


## 2. Prepare Aircraft Locations for Weather Extraction

The raw OpenSky response remains unchanged. A temporary DataFrame is created only to identify aircraft locations that need weather data.


In [6]:
opensky_columns = [
    "icao24",
    "callsign",
    "origin_country",
    "time_position",
    "last_contact",
    "longitude",
    "latitude",
    "baro_altitude",
    "on_ground",
    "velocity",
    "true_track",
    "vertical_rate",
    "sensors",
    "geo_altitude",
    "squawk",
    "spi",
    "position_source",
    "category"
]

if states:
    aircraft_df = pd.DataFrame(states, columns=opensky_columns)

    aircraft_locations = (
        aircraft_df[
            [
                "icao24",
                "latitude",
                "longitude",
                "geo_altitude",
                "time_position"
            ]
        ]
        .dropna(subset=["latitude", "longitude"])
        .copy()
    )

    print("Aircraft with usable locations:", len(aircraft_locations))
    display(aircraft_locations.head())

else:
    aircraft_locations = pd.DataFrame()
    print("No aircraft locations available.")


Aircraft with usable locations: 131


,icao24,latitude,longitude,geo_altitude,time_position
0,739222,32.8457,35.0522,449.58,1789744835
1,74282d,32.0513,35.0496,6393.18,1789744835
2,8015c2,22.9726,51.5766,11414.76,1789744834
3,0180a0,23.6820,50.2697,11711.94,1789744834
4,728679,31.7282,36.0761,1059.18,1789744725


In [7]:
# Group nearby aircraft into approximate weather locations
# to reduce unnecessary weather API requests.

if not aircraft_locations.empty:
    aircraft_locations["weather_latitude"] = (
        aircraft_locations["latitude"].round(1)
    )

    aircraft_locations["weather_longitude"] = (
        aircraft_locations["longitude"].round(1)
    )

    weather_locations = (
        aircraft_locations[
            ["weather_latitude", "weather_longitude"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    print("Aircraft locations:", len(aircraft_locations))
    print("Unique weather locations:", len(weather_locations))

else:
    weather_locations = pd.DataFrame()
    print("No weather locations available.")


Aircraft locations: 131
Unique weather locations: 121


## 3. Extract Open-Meteo Weather Data

Weather is requested for aircraft-relevant locations rather than one fixed Riyadh location. Pressure-level variables are used so weather can later be matched to aircraft altitude.


In [8]:
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_variables = [
    "temperature_850hPa",
    "wind_speed_850hPa",
    "wind_direction_850hPa",
    "geopotential_height_850hPa",

    "temperature_700hPa",
    "wind_speed_700hPa",
    "wind_direction_700hPa",
    "geopotential_height_700hPa",

    "temperature_500hPa",
    "wind_speed_500hPa",
    "wind_direction_500hPa",
    "geopotential_height_500hPa",

    "temperature_300hPa",
    "wind_speed_300hPa",
    "wind_direction_300hPa",
    "geopotential_height_300hPa",

    "temperature_250hPa",
    "wind_speed_250hPa",
    "wind_direction_250hPa",
    "geopotential_height_250hPa",

    "temperature_200hPa",
    "wind_speed_200hPa",
    "wind_direction_200hPa",
    "geopotential_height_200hPa"
]


In [9]:
weather_data = None

if not weather_locations.empty:
    latitudes = ",".join(
        weather_locations["weather_latitude"].astype(str)
    )

    longitudes = ",".join(
        weather_locations["weather_longitude"].astype(str)
    )

    params = {
        "latitude": latitudes,
        "longitude": longitudes,
        "hourly": ",".join(weather_variables),
        "forecast_days": 1
    }

    start_time = perf_counter()

    try:
        response = requests.get(
            weather_url,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        weather_data = response.json()
        duration = perf_counter() - start_time

        weather_file = raw_path / f"openmeteo_{timestamp}.json"

        with open(weather_file, "w", encoding="utf-8") as f:
            json.dump(weather_data, f)

        file_size_mb = weather_file.stat().st_size / (1024 * 1024)

        print("Open-Meteo Status Code:", response.status_code)
        print("Weather locations requested:", len(weather_locations))
        print("File size:", round(file_size_mb, 3), "MB")
        print("Request duration:", round(duration, 2), "seconds")
        print("Open-Meteo data saved to:", weather_file)

        write_extraction_log(
            source="Open-Meteo",
            status="SUCCESS",
            rows=len(weather_locations),
            file_size_mb=round(file_size_mb, 3),
            duration_seconds=duration,
            status_code=response.status_code
        )

    except requests.exceptions.Timeout:
        duration = perf_counter() - start_time
        print("Open-Meteo request timed out.")

        write_extraction_log(
            source="Open-Meteo",
            status="FAILED",
            rows=0,
            file_size_mb=0,
            duration_seconds=duration,
            error_message="Request timed out"
        )

    except requests.exceptions.HTTPError as e:
        duration = perf_counter() - start_time
        status_code = response.status_code if "response" in locals() else None
        print("Open-Meteo HTTP error:", e)

        write_extraction_log(
            source="Open-Meteo",
            status="FAILED",
            rows=0,
            file_size_mb=0,
            duration_seconds=duration,
            status_code=status_code,
            error_message=str(e)
        )

    except requests.exceptions.RequestException as e:
        duration = perf_counter() - start_time
        print("Open-Meteo request failed:", e)

        write_extraction_log(
            source="Open-Meteo",
            status="FAILED",
            rows=0,
            file_size_mb=0,
            duration_seconds=duration,
            error_message=str(e)
        )

else:
    print(
        "Weather extraction skipped because "
        "no aircraft locations were available."
    )


Open-Meteo Status Code: 200
Weather locations requested: 121
File size: 0.683 MB
Request duration: 1.39 seconds
Open-Meteo data saved to: c:\Users\W 10 Pro\Downloads\DE-SkyPrint-main\DE-SkyPrint-main\data\raw\openmeteo_2026-09-18_15-00-19.json


## 4. Check Aircraft Reference Data

The aircraft database is a reference dataset, so it does not need to be downloaded on every live extraction run.


In [10]:
aircraft_file = raw_path / "aircraftDatabase.csv"

if aircraft_file.exists():
    aircraft_file_size_mb = (
        aircraft_file.stat().st_size / (1024 * 1024)
    )

    print("Aircraft database is available.")
    print("File size:", round(aircraft_file_size_mb, 3), "MB")

else:
    print(
        "WARNING: aircraftDatabase.csv "
        "was not found in data/raw/"
    )


Aircraft database is available.
File size: 90.131 MB


## 5. Review Extraction Log


In [11]:
print("Extraction Log:")

if log_file.exists():
    extraction_log = pd.read_csv(log_file)
    display(extraction_log.tail(10))
else:
    print("No extraction log found.")


Extraction Log:


,extraction_time_utc,source,status,rows_extracted,file_size_mb,duration_seconds,status_code,error_message
0,2026-09-18T15:00:19.613684+00:00,OpenSky,SUCCESS,131,0.019,0.88,200,NaN
1,2026-09-18T15:00:19.613684+00:00,Open-Meteo,SUCCESS,121,0.683,1.39,200,NaN


## 6. List Raw Files and Sizes


In [12]:
print("Files currently inside data/raw:")

for file in sorted(raw_path.iterdir()):
    if file.is_file():
        file_size_mb = file.stat().st_size / (1024 * 1024)

        print(
            "-",
            file.name,
            "|",
            round(file_size_mb, 3),
            "MB"
        )


Files currently inside data/raw:
- aircraftDatabase.csv | 90.131 MB
- openmeteo_2026-09-12.json | 0.008 MB
- openmeteo_2026-09-18_15-00-19.json | 0.683 MB
- opensky_2026-09-12.json | 1.24 MB
- opensky_2026-09-18_15-00-19.json | 0.019 MB
